In [22]:
import pdfplumber
import fitz  # PyMuPDF
import os
import re
import docx
from docx import Document
#path to the pdf 
def extract_from_pdf_pdfplumber(file_path: str) -> str:
    """Extract text using pdfplumber (primary method)."""
    text_chunks = []

    with pdfplumber.open(file_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text(x_tolerance=2)
            if text:
                text_chunks.append(text)

    return "\n".join(text_chunks)

def extract_from_pdf_pymupdf(file_path: str) -> str:
    """Fallback PDF extraction using PyMuPDF."""
    text_chunks = []

    with fitz.open(file_path) as doc:
        for page in doc:
            text = page.get_text("text")
            if text:
                text_chunks.append(text)

    return "\n".join(text_chunks)

def extract_from_docx(file_path: str) -> str:
    """Extract text from .docx file."""
    doc = Document(file_path)
    paragraphs = [para.text for para in doc.paragraphs if para.text.strip()]
    return "\n".join(paragraphs)



In [17]:
def clean_text(text: str) -> str:
    """
    Clean extracted resume text.
    Removes weird bullets, excessive whitespace, and hidden characters.
    """
    if not text:
        return ""

    # Normalize unicode quotes/dashes
    text = text.replace("\u2013", "-")
    text = text.replace("\u2014", "-")
    text = text.replace("\u2022", "-")  # bullet → dash
    text = text.replace("\xa0", " ")    # non-breaking space

    # Remove control characters
    text = re.sub(r"[\x00-\x1f\x7f-\x9f]", "", text)

    # Remove multiple spaces
    text = re.sub(r"[ ]{2,}", " ", text)

    # Remove excessive newlines
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

In [18]:
def extract_text(file_path: str) -> str:
    """
    Main function:
    Detects file type and extracts raw text.
    """

    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")

    _, ext = os.path.splitext(file_path)
    ext = ext.lower()

    raw_text = ""

    if ext == ".pdf":
        try:
            raw_text = extract_from_pdf_pdfplumber(file_path)
            if not raw_text.strip():
                raise ValueError("pdfplumber returned empty text")
        except Exception:
            raw_text = extract_from_pdf_pymupdf(file_path)

    elif ext == ".docx":
        raw_text = extract_from_docx(file_path)

    elif ext == ".txt":
        with open(file_path, "r", encoding="utf-8") as f:
            raw_text = f.read()

    else:
        raise ValueError(f"Unsupported file format: {ext}")

    return clean_text(raw_text)

In [23]:
#file_path = "D:/cooding/ml/Mini-Project/CV_to_Portfolio_web/sample.pdf"
file_path = "D:/cooding/ml/Mini-Project/CV_to_Portfolio_web/aman_resume (2).docx"
text = extract_text(file_path)  
print(text)

About MeAman Yadav+91 9569491993 - amanstorm77@gmail.com - GitHub - LinkedInB.Tech in Mathematics & Computing student with hands-on experience building and deploying end-to-end machine learning solutions using Python, PyTorch, and Scikit-learn. Developed production-ready applications including rec- ommendation systems and healthcare risk prediction models, applying strong foundations in mathematics, statistics, and model evaluation.SkillsProgramming: Python, C++, MATLAB, R, SQLMachine Learning: Feature Engineering, Feature Analysis, Model Training, Model Evaluation, Scikit-learn, Regression, Classification, Cross-Validation, NLPData Analysis: Pandas, NumPy, Statistical Analysis, Exploratory Data Analysis (EDA)Deployment & Tools: Streamlit, Git.Work ExperienceProject-Based Experience2023 - PresentBuilt a CNN-LSTM model to predict Remaining Useful Life (RUL) of EV batteries using NASA Li-ion battery dataset, achieving low MAE and stable prediction performance across multiple charge cycle

### First implementing NLP using Spacy  - a custom NER model ;
this code snippit also generates a fake training data 

In [5]:
import re

def extract_project_section(resume_text):
    # 1. Define common major section headings
    # The ^ ensures we only match these words if they appear at the START of a line
    # (so we don't accidentally split on the word "skills" in a sentence like "Used my programming skills to...")
    heading_pattern = r'(?im)^(experience|work experience|education|skills|core competencies|technical skills|certifications|languages|projects|personal projects|academic projects|summary|profile)\b:?'

    # 2. Find all headings and their start/end positions in the text
    headings = []
    for match in re.finditer(heading_pattern, resume_text):
        headings.append({
            "name": match.group(1).lower(),
            "start": match.start(),
            "end": match.end()
        })

    # 3. Locate the "Projects" heading in our list of found headings
    project_index = -1
    for i, heading in enumerate(headings):
        if "project" in heading["name"]:
            project_index = i
            break

    if project_index == -1:
        return "No Project section found."

    # 4. Determine where the text block starts and ends
    project_start_idx = headings[project_index]["end"]
    
    # If there is another heading AFTER "Projects", stop there. 
    # Otherwise, grab everything until the end of the document.
    if project_index + 1 < len(headings):
        next_heading_start_idx = headings[project_index + 1]["start"]
    else:
        next_heading_start_idx = len(resume_text)

    # 5. Extract and clean the text block
    project_text = resume_text[project_start_idx:next_heading_start_idx].strip()
    
    # Strip out any trailing colons, dashes, or empty lines left over
    project_text = re.sub(r'^[:\-\s]+', '', project_text)

    return project_text

# --- Example Usage ---
mock_resume = """
Aman Yadav
Email: aman@email.com

Skills:
Python, PyTorch, React, SQL

Projects:
Automated Portfolio Builder
- Developed a web app using Python and React.
- Integrated a custom spaCy NER pipeline.

Education:
B.Tech in Mathematics and Computing
"""

extracted_projects = extract_project_section(mock_resume)
print("--- Extracted Project Section ---")
print(extracted_projects)

--- Extracted Project Section ---
Automated Portfolio Builder
- Developed a web app using Python and React.
- Integrated a custom spaCy NER pipeline.


In [4]:

import spacy
from spacy.training.example import Example
import random
from faker import Faker
import warnings

# Suppress minor spaCy warnings for a cleaner console output
warnings.filterwarnings("ignore", category=UserWarning)

fake = Faker()

# ==========================================
# 1. SYNTHETIC DATA GENERATOR SETUP
# ==========================================

SKILLS_LIST = [
    "Python", "PyTorch", "TensorFlow", "Scikit-learn", "Keras", "Pandas", "NumPy", 
    "Machine Learning", "Deep Learning", "NLP", "Computer Vision", "YOLO", "DeepLabV3", 
    "OpenCV", "Transformers", "LLMs", "Data Analysis", "Statistical Modeling",
    "C++", "Java", "JavaScript", "TypeScript", "React", "Node.js", "Express", 
    "HTML5", "CSS3", "Tailwind", "SQL", "PostgreSQL", "MongoDB", "Docker", 
    "Kubernetes", "AWS", "GCP", "Azure", "Git", "Streamlit", "Linux"
]

JOB_TITLES = [
    "Software Engineer", "Senior Software Engineer", "Data Scientist", 
    "Machine Learning Engineer", "Computer Vision Engineer", "NLP Researcher", 
    "Frontend Developer", "Backend Developer", "Full Stack Developer", 
    "Data Analyst", "AI Specialist", "Cloud Architect", "DevOps Engineer",
    "Quantitative Analyst", "Research Scientist", "Web Developer"
]

COMPANIES = [
    "Google", "Amazon", "Microsoft", "Meta", "Apple", "Netflix", 
    "TCS", "Infosys", "Wipro", "Accenture", "IBM", "Intel", "NVIDIA",
    "OpenAI", "Hugging Face", "TechNova Solutions", "DataRiver Inc.", 
    "Quantum Dynamics", "StartupX", "Global Finance Corp"
]

DEGREES = [
    "B.Tech in Computer Science", "B.Tech in Mathematics and Computing",
    "M.S. in Data Science", "M.S. in Artificial Intelligence", 
    "B.S. in Software Engineering", "BCA", "MCA", 
    "B.S. in Physics", "Ph.D. in Computer Vision", "Ph.D. in Machine Learning"
]

UNIVERSITIES = [
    "MIT", "Stanford University", "Carnegie Mellon University", "UC Berkeley",
    "Central University of Karnataka", "IIT Bombay", "IIT Delhi", "IIT Madras",
    "BITS Pilani", "University of Toronto", "University of Oxford", 
    "State University", "National Institute of Technology", "Tech Institute"
]

EMAIL_PREFIXES = ["Email: ", "E: ", "Contact: ", ""]
PHONE_PREFIXES = ["Phone: ", "P: ", "Mob: ", "+91 ", "+1 ", ""]
SKILL_PREFIXES = ["Skills: ", "Core Competencies: ", "Technical Skills: ", "Expertise: "]

def generate_resume_sample():
    text = ""
    entities = []

    # NAME
    name = fake.name()
    start = len(text)
    text += name + "\n"
    entities.append((start, start + len(name), "NAME"))

    # EMAIL
    email_pref = random.choice(EMAIL_PREFIXES)
    text += email_pref
    email = fake.email()
    start = len(text)
    text += f"{email} | "
    entities.append((start, start + len(email), "EMAIL"))

    # PHONE
    phone_pref = random.choice(PHONE_PREFIXES)
    text += phone_pref
    phone = "".join([str(random.randint(0, 9)) for _ in range(10)]) 
    if random.choice([True, False]):
        phone = f"{phone[:3]}-{phone[3:6]}-{phone[6:]}" 
    start = len(text)
    text += f"{phone}\n"
    entities.append((start, start + len(phone), "PHONE"))

    # GITHUB
    github = f"github.com/{name.replace(' ', '').lower()}{random.randint(1,99)}"
    text += "GitHub: "
    start = len(text)
    text += f"{github} | "
    entities.append((start, start + len(github), "GITHUB"))

    # LINKEDIN
    linkedin = f"linkedin.com/in/{name.replace(' ', '').lower()}{random.randint(1,99)}\n"
    text += "LinkedIn: "
    start = len(text)
    text += linkedin
    entities.append((start, start + len(linkedin.strip()), "LINKEDIN"))

    # SKILLS
    text += random.choice(SKILL_PREFIXES)
    user_skills = random.sample(SKILLS_LIST, random.randint(4, 8))
    for i, skill in enumerate(user_skills):
        start = len(text)
        text += skill
        entities.append((start, start + len(skill), "SKILLS"))
        if i < len(user_skills) - 1:
            text += random.choice([", ", " | ", " • "]) 
    text += "\n"

    # EXPERIENCE
    text += "Experience:\n"
    job = random.choice(JOB_TITLES)
    start = len(text)
    text += job
    entities.append((start, start + len(job), "JOB_TITLE"))
    
    text += " at "
    company = random.choice(COMPANIES)
    start = len(text)
    text += company + "\n"
    entities.append((start, start + len(company), "COMPANY"))

    # EDUCATION
    text += "Education:\n"
    degree = random.choice(DEGREES)
    start = len(text)
    text += degree
    entities.append((start, start + len(degree), "EDUCATION"))

    text += random.choice([" from ", ", "])
    uni = random.choice(UNIVERSITIES)
    start = len(text)
    text += uni + "\n"
    entities.append((start, start + len(uni), "UNIVERSITY"))

    return (text, {"entities": entities})


# ==========================================
# 2. GENERATE TRAINING DATA
# ==========================================
print("Generating 150 training samples...")
TRAIN_DATA = [generate_resume_sample() for _ in range(150)]


# ==========================================
# 3. SPACY MODEL SETUP & TRAINING
# ==========================================
print("Initializing blank English model...")
nlp = spacy.blank("en")
ner = nlp.add_pipe("ner")

# Add the exact labels we used in the generator
labels = [
    "NAME", "EMAIL", "PHONE", "GITHUB", "LINKEDIN", 
    "SKILLS", "JOB_TITLE", "COMPANY", "EDUCATION", "UNIVERSITY"
]

for label in labels:
    ner.add_label(label)

# Begin Training
optimizer = nlp.begin_training()

print("\nStarting Training Loop (30 Epochs)...")
for epoch in range(30):
    random.shuffle(TRAIN_DATA)
    losses = {}

    for text, annotations in TRAIN_DATA:
        doc = nlp.make_doc(text)
        example = Example.from_dict(doc, annotations)
        # Update the model, drop specifies the dropout rate to prevent overfitting
        nlp.update([example], drop=0.2, losses=losses)

    print(f"Epoch {epoch + 1:02d} - Loss: {losses['ner']:.2f}")


# ==========================================
# 4. SAVE & TEST THE MODEL
# ==========================================
output_dir = "portfolio_ner_model"
nlp.to_disk(output_dir)
print(f"\nModel successfully saved to directory: '{output_dir}'")

# Test on a brand new string
print("\n--- Testing Model on Unseen Data ---")
test_text = """
Rahul Sharma
Email: rahul.s@webmail.com | Mob: 9876543210
GitHub: github.com/rahuls
Technical Skills: Deep Learning, PyTorch, React, Node.js
Experience: NLP Researcher at OpenAI
Education: B.Tech in Mathematics and Computing, IIT Delhi
"""

# Load the saved model (this is how you'll load it in your website backend later)
loaded_nlp = spacy.load(output_dir)
doc = loaded_nlp(test_text)

for ent in doc.ents:
    print(f"{ent.label_:12} -> {ent.text}")


Generating 150 training samples...
Initializing blank English model...

Starting Training Loop (30 Epochs)...
Epoch 01 - Loss: 2263.95
Epoch 02 - Loss: 86.87
Epoch 03 - Loss: 17.34
Epoch 04 - Loss: 5.61
Epoch 05 - Loss: 0.48
Epoch 06 - Loss: 0.11
Epoch 07 - Loss: 0.01
Epoch 08 - Loss: 0.09
Epoch 09 - Loss: 0.00
Epoch 10 - Loss: 3.81
Epoch 11 - Loss: 14.05
Epoch 12 - Loss: 29.85
Epoch 13 - Loss: 21.44
Epoch 14 - Loss: 26.17
Epoch 15 - Loss: 15.23
Epoch 16 - Loss: 0.10
Epoch 17 - Loss: 2.03
Epoch 18 - Loss: 3.46
Epoch 19 - Loss: 2.16
Epoch 20 - Loss: 1.69
Epoch 21 - Loss: 0.00
Epoch 22 - Loss: 0.00
Epoch 23 - Loss: 0.00
Epoch 24 - Loss: 0.00
Epoch 25 - Loss: 0.00
Epoch 26 - Loss: 0.00
Epoch 27 - Loss: 47.33
Epoch 28 - Loss: 49.84
Epoch 29 - Loss: 5.48
Epoch 30 - Loss: 0.37

Model successfully saved to directory: 'portfolio_ner_model'

--- Testing Model on Unseen Data ---
JOB_TITLE    -> Rahul Sharma
EMAIL        -> rahul.s@webmail.com
PHONE        -> 9876543210
LINKEDIN     -> github.com